# Efficient ResNet Challenge — CIFAR-10 starter (≤ 5M params)

**Before you run:** `Runtime → Change runtime type → GPU` (T4 is fine).

Pipeline:
1. Download competition files from Kaggle (for the 2,000 test images + sample submission)
2. Load the 50k CIFAR-10 training images (torchvision), hold out a validation split
3. Wide pre-activation ResNet-18 variant, **~4.8M params** (asserted below)
4. Fast training: all data on GPU, GPU-side augmentation (crop + flip + cutout), mixed precision, OneCycle LR, label smoothing
5. Predict test set with flip-TTA → `submission.csv`

Recommended workflow: run with `VAL_SIZE = 5000` while experimenting, then set `VAL_SIZE = 0` and retrain on all 50k for your final submission.

In [ ]:
!nvidia-smi -L
import os, glob, time, math, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import torchvision

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
assert device == 'cuda', 'Switch the runtime to GPU: Runtime > Change runtime type'

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

## 1. Get the competition files from Kaggle

1. On the competition page, click **Join / I Understand and Accept** the rules (downloads fail otherwise).
2. Kaggle → Settings → **API → Create New Token** → downloads `kaggle.json`.
3. Set `COMP` to the slug in the competition URL: `kaggle.com/competitions/<slug>`.

In [ ]:
COMP = 'your-competition-slug'   # <-- CHANGE ME
DATA_DIR = Path('/content/data')

if not Path('~/.kaggle/kaggle.json').expanduser().exists():
    from google.colab import files
    print('Upload your kaggle.json')
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

if not DATA_DIR.exists():
    !kaggle competitions download -c {COMP} -p /content -q
    !unzip -q -o /content/{COMP}.zip -d {DATA_DIR}

# Show the layout so you know what's where
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_dir():
        print(f'{p}/  ({len(list(p.glob("*.png")))} png)')
print([p.name for p in DATA_DIR.rglob('*.csv')])

## 2. Load data

Training uses the standard CIFAR-10 50k training set via torchvision (the permitted training data).
If your course wants you to train from the competition's own training PNGs instead, swap in a loader for those — the rest of the notebook is unchanged.

In [ ]:
# ---- training data: 50k CIFAR-10 train images ----
tv = torchvision.datasets.CIFAR10('/content/cifar', train=True, download=True)
X_all = torch.tensor(tv.data).permute(0, 3, 1, 2).contiguous()   # N,3,32,32 uint8
y_all = torch.tensor(tv.targets)
CLASSES = tv.classes
print(X_all.shape, y_all.bincount().tolist())

# ---- test data: competition PNGs, ordered as in sample_submission ----
sample = pd.read_csv(next(DATA_DIR.rglob('sample_submission.csv')))
TEST_DIR = None   # set manually, e.g. DATA_DIR/'test', if auto-detect picks the wrong folder
if TEST_DIR is None:
    dirs = {}
    for p in DATA_DIR.rglob('*.png'):
        dirs.setdefault(p.parent, []).append(p)
    cand = [d for d, ps in dirs.items() if len(ps) == len(sample)] or \
           [d for d in dirs if 'test' in str(d).lower()]
    TEST_DIR = cand[0]
print('TEST_DIR =', TEST_DIR)

paths = {p.stem: p for p in Path(TEST_DIR).glob('*.png')}
X_test = torch.stack([
    torch.from_numpy(np.array(Image.open(paths[str(i)]).convert('RGB'))).permute(2, 0, 1)
    for i in sample['id']])
print('test:', X_test.shape)

In [ ]:
# Sanity check: look at a few training and test images
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for k in range(8):
    ax[0, k].imshow(X_all[k].permute(1, 2, 0)); ax[0, k].set_title(CLASSES[y_all[k]], fontsize=8)
    ax[1, k].imshow(X_test[k].permute(1, 2, 0)); ax[1, k].set_title(f'test id {sample.id[k]}', fontsize=8)
for a in ax.flat: a.axis('off')
plt.show()

## 3. Train/val split, GPU augmentation

In [ ]:
VAL_SIZE = 5000          # set to 0 for the final run on all 50k images

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
val_idx, tr_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]

MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)
def to_gpu(x):  # uint8 -> normalized float on GPU
    return ((x.to(device).float() / 255) - MEAN) / STD

Xtr, ytr = to_gpu(X_all[tr_idx]), y_all[tr_idx].to(device)
Xva, yva = to_gpu(X_all[val_idx]), y_all[val_idx].to(device)
Xte = to_gpu(X_test)
print('train', Xtr.shape, '| val', Xva.shape, '| test', Xte.shape)

def augment(x, pad=4, cutout=16):
    # Random crop (reflect pad), horizontal flip, cutout — all on GPU, per image.
    N, C, H, W = x.shape
    xp = F.pad(x, (pad,) * 4, mode='reflect')
    i = torch.randint(0, 2 * pad + 1, (N,), device=x.device)
    j = torch.randint(0, 2 * pad + 1, (N,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None].expand(N, C, H, W)
    cols = (j[:, None] + ar)[:, None, None, :].expand(N, C, H, W)
    x = xp[torch.arange(N, device=x.device)[:, None, None, None],
           torch.arange(C, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(N, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    if cutout:
        cy = torch.randint(0, H, (N,), device=x.device)[:, None, None]
        cx = torch.randint(0, W, (N,), device=x.device)[:, None, None]
        yy, xx = ar[None, :, None], ar[None, None, :]
        m = ((yy - cy).abs() < cutout // 2) & ((xx - cx).abs() < cutout // 2)
        x = x * (~m)[:, None].float()
    return x

## 4. Model: wide pre-activation ResNet (≤ 5M params)

`F(x)` = BN → ReLU → Conv3×3 → BN → ReLU → Conv3×3; `S(x)` = identity, or a 1×1 conv projection when shape changes.
Knobs to experiment with: `widths`, `blocks`, kernel size, dropout before the classifier.

In [ ]:
class PreActBlock(nn.Module):
    def __init__(self, cin, cout, stride=1, k=3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, k, stride, k // 2, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, k, 1, k // 2, bias=False)
        self.shortcut = None
        if stride != 1 or cin != cout:
            self.shortcut = nn.Conv2d(cin, cout, 1, stride, bias=False)   # learned projection S(x)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        s = self.shortcut(out) if self.shortcut is not None else x       # identity S(x)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + s                                                   # F(x) + S(x)

class ResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 352), blocks=(2, 2, 2, 1), num_classes=10, drop=0.0):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        layers, cin = [], widths[0]
        for s, (w, n) in enumerate(zip(widths, blocks)):
            for b in range(n):
                layers.append(PreActBlock(cin, w, stride=2 if (b == 0 and s > 0) else 1))
                cin = w
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.relu(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(self.drop(x))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

model = ResNet().to(device).to(memory_format=torch.channels_last)
n_params = count_params(model)
print(f'Trainable parameters: {n_params:,}')
assert n_params <= 5_000_000, 'Over the 5M budget!'

## 5. Train

Defaults: SGD + Nesterov, OneCycle LR, label smoothing 0.1, weight decay 5e-4, bf16/fp16 autocast.
On a T4 an epoch takes roughly 25–40 s, so 100 epochs is about an hour. More epochs (150–200) usually adds a bit more accuracy.

In [ ]:
EPOCHS     = 100
BATCH      = 256
LR         = 0.2
WD         = 5e-4
SMOOTHING  = 0.1

opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, nesterov=True, weight_decay=WD)
steps_per_epoch = math.ceil(len(Xtr) / BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps_per_epoch,
                                            pct_start=0.15, div_factor=10, final_div_factor=200)
criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
scaler = torch.amp.GradScaler('cuda')

@torch.no_grad()
def predict_logits(m, X, bs=1000, tta=True):
    m.eval(); out = []
    for k in range(0, len(X), bs):
        xb = X[k:k + bs].contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16):
            lo = m(xb).float()
            if tta: lo = lo + m(xb.flip(3)).float()
        out.append(lo)
    return torch.cat(out)

history = []
for epoch in range(EPOCHS):
    model.train(); t0 = time.time()
    order = torch.randperm(len(Xtr), device=device)
    tot_loss, correct = 0.0, 0
    for k in range(0, len(Xtr), BATCH):
        idx = order[k:k + BATCH]
        xb = augment(Xtr[idx]).contiguous(memory_format=torch.channels_last)
        yb = ytr[idx]
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tot_loss += loss.item() * len(idx)
        correct += (logits.argmax(1) == yb).sum().item()
    row = dict(epoch=epoch + 1, loss=tot_loss / len(Xtr), train_acc=correct / len(Xtr))
    if VAL_SIZE and ((epoch + 1) % 5 == 0 or epoch + 1 == EPOCHS):
        row['val_acc'] = (predict_logits(model, Xva).argmax(1) == yva).float().mean().item()
    history.append(row)
    print(f"ep {row['epoch']:3d} | loss {row['loss']:.3f} | train {row['train_acc']:.4f}"
          + (f" | val {row['val_acc']:.4f}" if 'val_acc' in row else '') + f" | {time.time() - t0:.1f}s")

torch.save(model.state_dict(), '/content/model.pt')

In [ ]:
h = pd.DataFrame(history)
ax = h.plot(x='epoch', y=['train_acc'] + (['val_acc'] if 'val_acc' in h else []), marker='.', figsize=(8, 4))
ax.set_ylim(0.5, 1); ax.grid(alpha=.3); plt.show()

if VAL_SIZE:
    from sklearn.metrics import confusion_matrix
    pred = predict_logits(model, Xva).argmax(1).cpu()
    cm = confusion_matrix(yva.cpu(), pred)
    print('Val accuracy (flip TTA):', (pred == yva.cpu()).float().mean().item())
    print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))

## 6. Predict & write `submission.csv`

In [ ]:
test_pred = predict_logits(model, Xte).argmax(1).cpu().numpy()
sub = pd.DataFrame({'id': sample['id'], 'label': test_pred})
assert len(sub) == len(sample) and sub['label'].between(0, 9).all()
sub.to_csv('/content/submission.csv', index=False)
print(sub.head()); print(sub['label'].value_counts().sort_index().tolist())

# Option A: download to your computer
from google.colab import files; files.download('/content/submission.csv')
# Option B: submit straight from Colab
# !kaggle competitions submit -c {COMP} -f /content/submission.csv -m "preact-resnet 4.8M, {EPOCHS} ep"

## Ideas to try next
- **Final run:** `VAL_SIZE = 0`, 150–200 epochs, same hyperparameters.
- **Augmentation:** add CutMix/MixUp, or TrivialAugment (more epochs needed when augmenting harder).
- **Architecture:** trade depth for width within budget, e.g. `widths=(96,192,320), blocks=(2,2,2)` (3 wide stages, ~4.9M) — always check the param count cell.
- **Weight averaging:** an EMA of model weights for the final epochs often adds ~0.2–0.5%.
- **Ensembling / distillation:** train 2–3 seeds and distill into one ≤5M student (an ensemble itself would exceed the budget).
- Log every experiment (config + val acc) — you'll need it for the report.